# Dedicated MX E4M3/A32 — honderd-epochreferentie

Dit notebook analyseert **uitsluitend opgeslagen resultaten** van de dedicated E4M3-gewichtsreferentie: CNN-8/SVHN train-only, A32-activaties en een FP32-updatemaster. Er wordt geen training gestart en geen checkpoint geladen.

**Epoch100 is het vaste hoofdvergelijkingspunt.** Het beste E4M3-checkpoint wordt ernaast getoond; de laatste beschikbare epoch blijft afzonderlijk zichtbaar. Bij een onderbroken run wordt geen voltooiing of eindaccuracy verondersteld. Synthetische controlegegevens zijn bovenaan en in iedere figuurtitel herkenbaar gemarkeerd.


In [ ]:
from pathlib import Path
RUN_DIR = Path('.')  # In de gekopieerde run-notebook automatisch het exacte resultaatpad.
SAVE_FIGURES = False  # True: PNG's naar een NIEUWE analysis_<tijdstip>-map; ruwe resultaten blijven behouden.


In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:  # Ook analyse vanuit een gewone Python-omgeving blijft mogelijk.
    class Markdown(str):
        pass
    def display(value):
        print(value)

sys.path.insert(0, str(RUN_DIR.expanduser().resolve()))
import notebook_support as ns

run = ns.load_run(RUN_DIR)
figures = {}
display(Markdown(ns.status_markdown(run)))
display(ns.protocol_table(run))
display(Markdown('### Meldingen en ontbrekende gegevens'))
display(ns.diagnostics_table(run))


## Epoch100, beste checkpoint en laatste beschikbare epoch

**Beste checkpoint:** het eerste strikte maximum van de opgeslagen E4M3-testaccuracy. Een gelijke score vervangt het eerdere beste checkpoint niet. Alle scores komen uit dezelfde dedicated run. De selectiebron en checkpointnaam staan expliciet in de tabel; een ontbrekend selectiebestand wordt niet als bevestigde selectie behandeld.

De officiële testset wordt volgens het bestaande protocol gebruikt voor selectie. De beste score is daarom **geen onafhankelijke testschatting na selectie**. Epoch100 blijft het vaste hoofdvergelijkingspunt. Zolang de run niet volledig als PASS is afgerond, toont het notebook de laatste beschikbare epoch en de beste score tot dan, zonder een epoch100-resultaat te voorspellen.


In [ ]:
display(ns.checkpoint_table(run).round({'Testaccuracy [%]': 6, 'CE-loss': 8}))
figures['checkpoint_accuracy'] = ns.plot_checkpoint_accuracy(run)
display(figures['checkpoint_accuracy'])
plt.close(figures['checkpoint_accuracy'])


In [ ]:
display(Markdown('## Leercurves\n\nTrainaccuracy en trainloss zijn tijdens de updates verzameld; testmetrics zijn na de epoch met bevroren evaluatiestaat gemeten. De figuur toont uitsluitend gemeten epochs.'))
figures['learning_curves'] = ns.plot_learning(run)
display(figures['learning_curves'])
plt.close(figures['learning_curves'])


In [ ]:
display(Markdown('## Gemeten looptijd\n\nBij een volledige referentie toont dit notebook de gemeten som van alle honderd epochtijden. Bij een onvolledige run blijft een opgeslagen looptijdprojectie als schatting herkenbaar.'))
display(Markdown(ns.timing_markdown(run)))
figures['timing'] = ns.plot_timing(run)
display(figures['timing'])
plt.close(figures['timing'])


In [ ]:
display(Markdown('## GPU-geheugen\n\nToegewezen en gereserveerd geheugen zijn de opgeslagen pieken van de PyTorch CUDA-allocator. Dit is geen meting van alle GPU-processen en geen bewijs van compacte native MX-uitvoering. Ontbrekende metingen blijven ontbrekend.'))
figures['gpu_memory'] = ns.plot_memory(run)
display(figures['gpu_memory'])
plt.close(figures['gpu_memory'])


In [ ]:
display(Markdown('## Opgeslagen technische controles\n\nDeze tabel rapporteert bestaande controles uit de run; het notebook voert geen nieuwe gradient-, model- of checkpointcontrole uit.'))
display(ns.audit_table(run))
display(Markdown('### Gewichtsupdates per epoch en doellaag'))
display(ns.weight_update_table(run))


## Interpretatie en grenzen

Een volledige honderd-epochrun levert een dedicated E4M3/A32-referentie voor deze workload en seed. Dit experiment bevat geen distillatie of exponenttruncatie en toont daarom nog geen winst van exponenttruncatie aan. De bestaande mantissaresultaten en latere gezamenlijke experimenten houden hun eigen resultaten en protocol.

A32 en FP32 beschrijven de activatie- en masterdtypes. De **opgeslagen CUDA-backendprecisie staat apart in de protocoltabel**: de cuDNN-convoluties kunnen intern TF32 gebruiken. De emulator levert geen meting van native MX-snelheid of energie. Bij een onderbroken of mislukte run blijven beschikbare epochs zichtbaar, samen met de runstatus en meldingen.

Met `SAVE_FIGURES = True` worden alleen de vier figuren en een analysemanifest naar een **nieuwe** map geschreven. Een volgende uitvoering maakt opnieuw een nieuwe map. Geen JSON, CSV, checkpoint of eerdere figuur wordt overschreven.


In [ ]:
if SAVE_FIGURES:
    analysis_dir = ns.save_figures(figures, run)
    print('Figuren opgeslagen in:', analysis_dir)
else:
    print('Alleen analyse in het notebook; geen figuren naar schijf geschreven.')

print('Ingelezen resultaatmap:', run.path)
display(pd.DataFrame({'Ingelezen bestand': [path.name for path in run.input_files]}))
